#transform customer data

##remove null customer_id

In [0]:
df = spark.sql('select * from gizmobox.bronze.py_customers')
display(df)

In [0]:
df = spark.table('gizmobox.bronze.py_customers')
display(df)

In [0]:
df = spark.read.table('gizmobox.bronze.py_customers')
display(df)

##remove exact duplicate records

In [0]:
# df1 = df.filter('customer_id is not null')
# display(df1)
# or
# df1 = df1.filter(df.customer_id.isNotNull())
# display(df1)
df1 = df1.where(df.customer_id.isNotNull())
display(df1)

##exact duplicates drop

In [0]:
# df2 = df1.distinct()
# display(df2)
df2 = df1.dropDuplicates
display(df2)

##remove duplicate records based on created_timestamp

In [0]:
from pyspark.sql import functions as F
df_max_ts = df2().groupby('customer_id')\
            .agg(F.max('created_timestamp').alias('max_ts'))
display(df_max_ts)

##cast the correct data_types and joins
We can write using CTE

In [0]:
df2_instance = df2()

df_final = df2_instance.join(
    df_max_ts,
    (df2_instance.customer_id == df_max_ts.customer_id) & 
    (df2_instance.created_timestamp == df_max_ts.max_ts),
    'inner'
).select(df2_instance['*'])

display(df_final)

In [0]:
df_final_1 = df_final.select(df_final.created_timestamp.cast('timestamp'),
                           df_final.customer_id,
                           df_final.customer_name,
                           df_final.date_of_birth.cast('date'),
                           df_final.email,
                           df_final.member_since.cast('date'),
                           df_final.telephone)
display(df_final_1)

##write data to delta table

In [0]:
df_final_1.writeTo('Gizmobox.silver.py_customers').createOrReplace()

In [0]:
%sql
select * from Gizmobox.silver.py_customers